**Load Gdrive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Data Pipeline (Merge Dataset)**

https://www.kaggle.com/datasets/greegtitan/indonesia-climate

In [2]:
import pandas as pd

df_climate = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AI-CLIMATE/data/climate_data.csv")
df_station = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AI-CLIMATE/data/station_detail.csv")
df_province = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AI-CLIMATE/data/province_detail.csv")

# merge station
climate_station = pd.merge(df_climate, df_station, on="station_id")

# merge province
data = pd.merge(climate_station, df_province, on="province_id", how="left")

print(data.head())

         date    Tn    Tx  Tavg  RH_avg    RR   ss  ff_x  ddd_x  ff_avg  \
0  01-01-2010  21.4  30.2  27.1    82.0   9.0  0.5   7.0   90.0     5.0   
1  02-01-2010  21.0  29.6  25.7    95.0  24.0  0.2   6.0   90.0     4.0   
2  03-01-2010  20.2  26.8  24.5    98.0  63.0  0.0   5.0   90.0     4.0   
3  04-01-2010  21.0  29.2  25.8    90.0   0.0  0.1   4.0  225.0     3.0   
4  05-01-2010  21.2  30.0  26.7    90.0   2.0  0.4   NaN    NaN     NaN   

  ddd_car  station_id                      station_name  region_name  \
0      E        96001  Stasiun Meteorologi Maimun Saleh  Kota Sabang   
1      E        96001  Stasiun Meteorologi Maimun Saleh  Kota Sabang   
2      E        96001  Stasiun Meteorologi Maimun Saleh  Kota Sabang   
3      SW       96001  Stasiun Meteorologi Maimun Saleh  Kota Sabang   
4     NaN       96001  Stasiun Meteorologi Maimun Saleh  Kota Sabang   

   latitude  longitude  region_id  province_id             province_name  
0   5.87655   95.33785         20        

**Data Cleaning**

Tangani missing value.

In [3]:
import numpy as np

# mean
data['Tavg'].fillna(data['Tavg'].mean(), inplace=True)
data['RH_avg'].fillna(data['RH_avg'].mean(), inplace=True)
data['RR'].fillna(data['RR'].mean(), inplace=True)
data['ff_avg'].fillna(data['ff_avg'].mean(), inplace=True)

# median
data['Tn'].fillna(data['Tn'].median(), inplace=True)
data['Tx'].fillna(data['Tx'].median(), inplace=True)
data['ss'].fillna(data['ss'].median(), inplace=True)
data['ff_x'].fillna(data['ff_x'].median(), inplace=True)

/tmp/ipykernel_330/1535162039.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Tavg'].fillna(data['Tavg'].mean(), inplace=True)
/tmp/ipykernel_330/1535162039.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

**Feature Engineering**

Tambahkan feature waktu.

In [5]:
data['date'] = pd.to_datetime(data['date'], dayfirst=True)

data['year'] = data['date'].dt.year
data['month'] = data['date'].dt.month
data['day'] = data['date'].dt.day

**Split Dataset**

In [6]:
from sklearn.model_selection import train_test_split

X = data[['RH_avg','RR','ff_avg','ss','year','month','day']]
y = data['Tavg']

X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)

**Training Model (3 Model)**

Linear Regression

In [7]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train,y_train)

pred_lr = lr.predict(X_test)

Random Forest

In [8]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor()

rf.fit(X_train,y_train)

pred_rf = rf.predict(X_test)

XGBoost

In [9]:
from xgboost import XGBRegressor

xgb = XGBRegressor()

xgb.fit(X_train,y_train)

pred_xgb = xgb.predict(X_test)

**Bayesian Model Averaging**

Tujuan:

mengurangi bias model

meningkatkan stabilitas prediksi

In [10]:
final_pred = (pred_lr + pred_rf + pred_xgb) / 3

**Evaluasi Model**

In [11]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
import numpy as np

mae = mean_absolute_error(y_test,final_pred)
rmse = np.sqrt(mean_squared_error(y_test,final_pred))
r2 = r2_score(y_test,final_pred)

print("MAE:",mae)
print("RMSE:",rmse)
print("R2:",r2)

MAE: 1.015797152935059
RMSE: 1.6793193409273
R2: 0.22443138733351842


**1. MAE (Mean Absolute Error)**
MAE = 1.01

Artinya:

👉 Rata-rata kesalahan prediksi model sekitar 1.01°C

contoh:
| Suhu Asli | Prediksi Model |
| --------- | -------------- |
| 28°C      | 27°C           |
| 30°C      | 29°C           |

Rata-rata selisihnya sekitar 1 derajat.

| MAE   | Kualitas Model |
| ----- | -------------- |
| < 1   | sangat bagus   |
| 1 – 2 | cukup bagus    |
| > 3   | kurang bagus   |

jadi CUKUP BAIK

**2. RMSE (Root Mean Squared Error)**

RMSE = 1.67

RMSE memberi penalti lebih besar jika error besar.

contoh:
| Error | Dampak RMSE  |
| ----- | ------------ |
| 1°C   | kecil        |
| 5°C   | sangat besar |

Interpretasi:

👉 Model memiliki error sekitar 1.67°C

Jika dibandingkan:
| RMSE  | Interpretasi |
| ----- | ------------ |
| < 1   | sangat bagus |
| 1 – 2 | cukup bagus  |
| > 3   | buruk        |

jadi model CUKUP BAIK


**3. R² Score (Koefisien Determinasi)**
R2 = 0.224

R² mengukur seberapa baik model menjelaskan data.

Range:

0   = model sangat buruk
1   = model sempurna

Interpretasi umum:
| R²        | Kualitas Model |
| --------- | -------------- |
| < 0.3     | lemah          |
| 0.3 – 0.6 | sedang         |
| 0.6 – 0.8 | bagus          |
| > 0.8     | sangat bagus   |

Model:

R2 = 0.22

Artinya:

👉 Model hanya menjelaskan 22% pola data.

Masih:

⚠️ kurang kuat


---


**KESIMPULAN**
| Metric | Hasil  | Interpretasi |
| ------ | ------ | ------------ |
| MAE    | 1.01°C | cukup baik   |
| RMSE   | 1.67°C | cukup baik   |
| R²     | 0.22   | masih lemah  |

Artinya:

- model bisa memprediksi suhu
- tetapi belum menangkap pola iklim dengan baik

**Simpan Model**

In [12]:
import joblib

model = {
"lr":lr,
"rf":rf,
"xgb":xgb
}

joblib.dump(model,"climate_model.pkl")

['climate_model.pkl']